# Shin et al. (2025) from dandiset 000248: decoder reading (b) with cross-validation seed +200000

Dandiset [000248](https://dandiarchive.org/dandiset/000248/0.240502.2344) version 0.240502.2344.
This notebook repeats decoder reading (b) of `decoding_shin_2025.ipynb` (C chosen by an inner
threefold cross-validation) on every area with a further cross-validation seed, the seed of the
decoding notebook plus 200,000, for fig3c-npx-v1-inference and fig3d-npx-areas-inference.
`decoding_seed1_shin_2025.ipynb` does the same with seed +100000, and
`decoding_checks_shin_2025.ipynb` repeats reading (a) with both seeds. The two seeds are in two
notebooks so that each runs within the notebook limit.

| Claim | Tested here |
|---|---|
| fig3c-npx-v1-inference, fig3d-npx-areas-inference | seed only; outcomes in `decoding_shin_2025.ipynb` |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import warnings

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import stats

import illusion as il

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
N_JOBS = 4

started = time.time()
sessions = il.load_all(ccg=False)
print(f"loaded {len(sessions)} sessions of dandiset {il.DANDISET_ID} version {il.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

RS units per area and session, and ICwcfg1 trials of the decoded images.

In [ ]:
rows = []
for s in sessions:
    u, b = s["units"], s["blocks"]["ICwcfg1"]
    n = {c: int(np.sum(b["codes"] == c)) for c in [il.IC1, il.LC1, il.LC2, il.IC2, il.TRE1, il.TRE2, il.XRE1, il.XRE2]}
    rows.append({"subject": s["subject"], **{f"{a}_RS": int(((u.area == a) & u.rs).sum()) for a in il.AREAS},
                 "trials IC1/LC1/LC2/IC2": "/".join(str(n[c]) for c in [il.IC1, il.LC1, il.LC2, il.IC2]),
                 "TRE1/TRE2": f"{n[il.TRE1]}/{n[il.TRE2]}", "XRE1/XRE2": f"{n[il.XRE1]}/{n[il.XRE2]}"})
units_tab = pd.DataFrame(rows)
print(units_tab.to_string(index=False))
allR = np.concatenate([s["blocks"]["ICwcfg1"]["R"].ravel() for s in sessions])
print(f"IC response (spikes/s): min {allR.min():.1f}, median {np.median(allR):.1f}, max {allR.max():.1f}")
print("Session-areas with >= 50 RS units:", {a: int((units_tab[f"{a}_RS"] >= 50).sum()) for a in il.AREAS},
      "(paper 12, 8, 11, 10, 12, 11)")

The counts are those of the data check of `decoding_shin_2025.ipynb`.

## Decoder reading (b), seed +200000

The TRE decoder of `decoding_shin_2025.ipynb`, reading (b), on every (session, area) with at least
50 RS units, with the seed of that notebook plus 200,000.

In [ ]:
K = 2
t0 = time.time()
jobs = []
for i, s in enumerate(sessions):
    u = s["units"]
    for j, a in enumerate(il.AREAS):
        m = ((u.area == a) & u.rs).values
        if m.sum() < 50:
            continue
        b = s["blocks"]["ICwcfg1"]
        jobs.append((s["subject"], a, b["R"][:, m], b["codes"], 1000 * i + j + K * 100000))

def tre(subject, a, R, codes, seed):
    r = il.decode(R, codes, [il.IC1, il.LC1, il.LC2, il.IC2], [il.TRE1, il.TRE2], seed=seed, reading="b")
    score, icv, lcv = il.inference_score(r["probe"])
    return {"subject": subject, "area": a, "seed": seed, "acc": r["acc"], "tre_ic": icv, "tre_lc": lcv,
            "inference": score, "unconverged": r["n_unconverged"]}

dec = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(tre)(*j) for j in jobs))
print(f"{len(jobs)} decoders in {time.time() - t0:.0f} s; sessions per area:",
      dec.groupby("area").size().reindex(il.AREAS, fill_value=0).to_dict(), "; fits that did not converge:", int(dec.unconverged.sum()))

In [ ]:
x = dec[dec.area == "V1"]
print(f"V1, reading (b), seed +{K}00000: P(TRE -> IC) {x.tre_ic.mean():.3f}, P(TRE -> LC) {x.tre_lc.mean():.3f}, "
      f"accuracy {x.acc.mean():.3f}; IC > LC in {(x.tre_ic > x.tre_lc).sum()} of {len(x)} sessions; "
      f"signed-rank IC vs LC p = {il.signrank_p(x.tre_ic, x.tre_lc):.4g}; "
      f"IC within 0.51 +/- 0.051: {abs(x.tre_ic.mean() - 0.51) <= 0.051}, LC within 0.22 +/- 0.022: {abs(x.tre_lc.mean() - 0.22) <= 0.022}")

In [ ]:
paper_p = {"V1": 0.0132, "LM": 0.0156, "RL": 0.0244, "AL": 0.0039, "PM": 0.0342, "AM": 0.1748}
rows = []
for a in il.AREAS:
    y = dec[dec.area == a]
    rows.append({"area": a, "sessions": len(y), "IC": y.tre_ic.mean(), "LC": y.tre_lc.mean(), "inference_mean": y.inference.mean(),
                 "p": il.signrank_p(y.inference), "paper_p": paper_p[a]})
t = pd.DataFrame(rows)
t["sig_IC_above_LC"] = (t.p < 0.05) & (t.inference_mean > 0)
print(t.round(4).to_string(index=False))
sig = t[t.sig_IC_above_LC].area.tolist()
print(f"reading (b), seed +{K}00000: significant {sig} ({len(sig)} areas); three largest mean scores "
      f"{t.nlargest(3, 'inference_mean').area.tolist()}; three smallest p {t.nsmallest(3, 'p').area.tolist()}")

**Result:** in V1, P(TRE -> IC) is 0.514 and P(TRE -> LC) 0.215, both within tolerance, with IC above LC in 10 of 12 sessions and signed-rank p = 0.0068 (fig3c). Five areas decode TRE as IC significantly more than as LC: V1, LM, RL, AL and AM (AM p = 0.0186), with PM not significant (p = 0.052); LM, V1 and AL have the three largest mean scores. With this seed, reading (b) meets the fig3d criterion, where the seed of `decoding_shin_2025.ipynb` gives four areas (PM p = 0.077, AM p = 0.083).

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")

## Summary

| Claim | Result with this seed |
|---|---|
| fig3c-npx-v1-inference, reading (b), seed +200000 | both values within tolerance, p = 0.0068 |
| fig3d-npx-areas-inference, reading (b), seed +200000 | five areas significant (V1, LM, RL, AL, AM); LM, V1, AL the three largest |

The outcomes are recorded from `decoding_shin_2025.ipynb`; the attempt record proposes a change of
the fig3d outcome to reproduced from these seeds. How the decoder differs from the paper's is
stated in the summary of `decoding_shin_2025.ipynb`.